# Energy sets and cross sections

This notebook shows just a few energy sets and how to convert between them. There's not much else: the energy set is fully defined by the cross sections; picking a right one and obtaining cross sections crucially determines the accuracy of the multi-group simulation.

In [ ]:
import jax
import jax.numpy as jnp
jax.config.update("jax_platforms", "cpu")
jax.config.update("jax_enable_x64", True)

Two example energy sets. They are always from high-to-low for this code since this is more logical for downscattering-only problems (you can solve first energy group 0, then 1, etc. instead of going from highest to lowest)

In [ ]:
from jax_sn.energy_set.energy_set_examples import vitaminj

energy_set_1 = vitaminj()
energy_set_2 = jnp.logspace(jnp.log10(energy_set_1[0]), jnp.log10(energy_set_1[-1]), 20)

print(energy_set_1)

We can visualize these energy sets by simply plotting the energy group boundaries as a function of the index:

In [ ]:
import matplotlib.pyplot as plt
plt.stairs(energy_set_2, label="vitaminj")
plt.stairs(energy_set_1, label="vitaminj")
plt.yscale('log')
plt.xlabel("$g$")
_=plt.ylabel("Energy [eV]")

The VitaminJ group structure is not completely logarithmic: it has more resolution in the fast energy regime compared to the resonance regime.

## Cross sections

The energy set defines the multigroup equations:



$$\Omega \cdot \nabla \psi_g(\mathbf{r}, \Omega) + \sigma_g(\mathbf{r}, \Omega)\psi_g(\mathbf{r}, \Omega) = \sum_{\ell}\sum_{\tilde{g}}Y_{\ell}(\Omega)  \sigma^s_{\ell  g  \tilde{g}}(\mathbf{r})\int_{S^2}Y_{\ell} (\tilde{\Omega})\psi_{\tilde{g}}(\mathbf{r},\tilde{\Omega}) \,d \tilde{\Omega} \nonumber  + q_g(\mathbf{r}, \Omega) $$


where the multigroup cross sections are pre-computed by assuming a *guess* for the angular flux spectrum:


$$  \sigma_g(\mathbf{r}, \Omega) = \frac{\int \sigma(\mathbf{r}, E) \psi(\mathbf{r}, E , \Omega) \, d E }{\int_g \psi \, d E}$$

$$ \sigma^s_{\ell  g  \tilde{g}}(\mathbf{r}) = \frac{\int_{S^2}\int_g \int_{\tilde{g}} Y_{\ell}(\tilde{\Omega}) \sigma^s_{\ell}(\mathbf{r}, \tilde{E}\to E)\psi(\mathbf{r}, \tilde{E},\tilde{\Omega}) \, d \tilde{E} \,d \tilde{\Omega}\,d E  }{\int_{S^2}\int_{\tilde{g}}Y_{\ell}(\tilde{\Omega}) \psi(\mathbf{r},\tilde{\Omega}, \tilde{E})\, d \tilde{E} \,d \tilde{\Omega}} $$


If a guess is good, the multigroup equation solution will be good; if a guess is bad, the solution will not be accurate to the true solution at all. Also, if a very fine energy group structure is used, the error obtained in wrongly integrating it will not be as large.

Note that after obtaining the cross sections, the energy group boundaries actually do not appear anymore!

### In the code

The multigroup cross sections are in this code a simple structure: each element points to a material, which has its array of total cross sections $\sigma_g$ and the scattering cross section matrices $\sigma_{\ell g \tilde{g}}$.

An example 2-group water material with first order anisotropic scattering is below:

In [ ]:
from jax_sn.domain.cross_sections import CrossSectionData


water = CrossSectionData(
    total = jnp.array([15.2090, 24.4140]),
    scattering = jnp.array([
        [
            [5.5069, 0.0],
            [6.6227, 11.4080]
        ],
        [
            [1.158567, 0.0],
            [1.1234767, 2.4733]
        ]
    ])
)
print(water)

The shape of the scattering is [2,2,2]: two for $l=1$ anisotropic scattering, 2 for two groups outgoing and 2 for two groups ingoing. It is furthermore lower triangular: each incoming group (last index; column) can only scatter to itself or lower energy groups (first index; row). 

Normally, you do not hard-code them like this and you obtain them from software which automates the integrals above, including the guess of the flux spectrum.

The [OpenMC](https://docs.openmc.org/en/stable/) code can obtain the multigroup cross sections from a simulation in your exact (or similar) geometry, or the [NJOY](https://github.com/njoy/njoy2016) code can assume common spectral shapes.

In [FW-CADIS](../examples/fw_cadis.ipynb) we show how we obtain a breeding-blanket multigroup cross section set from OpenMC.

## Comparison between energy sets

After computing a solution to the transport equation, you obtain 

$$ \psi_g(\mathbf{r}, \Omega) = \int_g \psi(\mathbf{r}, E, \Omega) \, d E$$

which is the group-averaged flux. This is not directly comparable between energy sets because the boundaries are different.

You can normalise the fluxes by diving by the bin width or lethargy to obtain a visual comparison of the spectrum.

Another option is to map the group-averaged flux to the other energy set using a overlap matrix (basically, how much of the energy group is contained within energy groups of the different energy set). This is exact if the groups do share their boundaries but one is just more coarse.

In [ ]:
from jax_sn.energy_set import compute_overlap_matrix